> debug 断点跟踪：熟悉流程以及源码细节
- 控制流（pipeline / workflow）和数据流
    - 数据流：tensor shape 以及 meaning，输入输出

- Cmd/Ctrl-Shift-P → Remote-SSH: Connect to Host → yang (~/.ssh/config)
- 再 Cmd/Ctrl-Shift-P → Dev Containers: Attach to Running Container → 选 verl-qwen35 (running container)
- Open Folder → /workspace/verl-train,在该容器上下文里装上 Python 扩展
    - 以及 Ray Distributed Debugger 插件（anyscalecompute）
    - cursor 侧边栏：ray debugger
- `docker exec -it verl-qwen35 bash -lc 'pip install -q debugpy'`
    - 宿主机执行，容器内安装 

> verl: 单控制器 + ray，两种断点

- Driver 断点(主流程): `.vscode/launch.json`
    - verl.trainer.main_ppo: driver 进程
    - debuggy 到 driver
    - cursor 红点断点
- ray Worker 断点(模型 forward / actor 更新)
    - actor 前/反向、rollout、模型 forward，TaskRunnerV1 是个 Ray actor
    - 在关心的 worker 代码里插 breakpoint() —— 例如 actor 的 update_policy(verl/verl/workers/actor/dp_actor.py),或模型 forward。
        - 必须走代码 `breakpoint()` 的方式
    - 似乎 rlhfdataset（`from torch.utils.data import Dataset`）, custom reward fn 也是封到了 ray actor 里

### main_ppo

```
main_ppo.py: TaskRunnerV1.run(146)
  └ trainer.init(147) → _setup(159) → _init_dataloader(≈550)
       └ create_rl_dataset(trainer/ppo/utils.py:110)
            └ RLHFDataset.__init__(rl_dataset.py:72)   # 读 parquet / 建索引,建 dataloader 时跑一次
       RLHFDataset.__getitem__                          # 单样本 tokenize/图像处理/拼 prompt,迭代时调
```

#### RLHFDataset

- dataset/collate 这类纯 CPU 逻辑,用独立 debug_dataset.py + launch + 红点;
    - `__getitem__`: 多模态数据的组织
- breakpoint() 只在真实训练的 Ray actor/worker 里用,且要避开 datasets 多进程 / 后台无控制台的地方。

```python
from omegaconf import OmegaConf

from verl.utils import hf_processor, hf_tokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset

MODEL = "/workspace/models/Qwen3.5-2B"
PARQUET = "/workspace/verl-train/data/geo3k/train.parquet"

tokenizer = hf_tokenizer(MODEL)
processor = hf_processor(MODEL, use_fast=True)  # geo3k 多模态,必须

cfg = OmegaConf.create(
    {
        "prompt_key": "prompt",
        "image_key": "images",
        "max_prompt_length": 1024,
        "truncation": "error",
        "return_raw_chat": True,
        "filter_overlong_prompts": False,
        "shuffle": False,
    }
)

ds = RLHFDataset(PARQUET, tokenizer, cfg, processor=processor)
print("dataset len:", len(ds))

# 在 rl_dataset.py 的 __getitem__(387)或 _build_messages(299)打红点,下面这行会命中:
sample = ds[0]
print("sample keys:", list(sample.keys()))
```